# 🧠 02. Context Engineering — 프롬프트 조립 · 계층형 메모리 · 컨텍스트 압축

본 실습 노트북은 에이전트의 **컨텍스트 윈도우에 무엇을, 어떻게 넣고, 넘치지 않게 관리하는지**를 다룹니다.
**`app/middleware/prompt/`**, **`app/middleware/memory/`**, **`app/middleware/compaction/`** 세 패키지를 단계별로 직접 실행하며 학습합니다.

---

### 💡 Context Engineering의 네 가지 축

| 축 | 질문 | 구현 |
|:---|:---|:---|
| **Select (선택)** | 매 호출마다 무엇을 골라 컨텍스트에 넣을까? | **5-Layer Prompt Stack** (`PromptAssembler`): 불변/가변 영역을 나눠 Prompt Caching 보존<br>**L4 Memory 주입**: 장기 기억 스냅샷 + FTS5로 고른 과거 세션 힌트, 필요할 때만 `session_recall`로 원문 인출 |
| **Write (기록)** | 컨텍스트 윈도우 밖에 무엇을 남겨 둘까? | **Semantic Memory** (`memory` 도구로 `MEMORY.md`, `USER.md`에 기록) + **Episodic Memory** (세션 요약·키워드를 SQLite FTS5에 색인) |
| **Compress (압축)** | 컨텍스트가 넘치기 전에 어떻게 줄일까? | **Tool Result Swap** (도구로 입력되는 대형 컨텍스트 → 디스크) + **SummarizationMiddleware** (LangChain prebuilt, 오래된 대화 → 요약) |
| **Isolate (격리)** | 서로 다른 작업·사용자의 컨텍스트를 어떻게 나눌까? | **Subagent 패턴**: 하위 에이전트가 독립된 컨텍스트에서 일하고 결과만 돌려줌 (👉 **1번 노트북 *Reasoning and Subagent***에서 학습)<br>이 노트북 6-7에서는 **user_id별 메모리·워크스페이스 격리**를 확인합니다. |

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 세부 실행 셀 |
|:---:|:---|:---|
| **Step 0** | **환경 세팅 & 격리 샌드박스** | 루트 탐색, `.env` 로드, `nest_asyncio`, 실습 격리 샌드박스(`demo_dir`) 생성 |
| **Part 1** | **5-Layer Prompt Stack** | 1-1) 도구 및 스킬 카탈로그 빌더 준비<br>1-2) 5-Layer 전체 시스템 프롬프트 조립 & 덤프 출력 |
| **Part 2** | **Semantic Memory (장기 기억)** | 2-1) `§` 엔트리별 프로필 조회<br>2-2) 새 엔트리 추가 & 중복 차단<br>2-3) 특정 엔트리 단위 교체(`replace`)<br>2-4) 시스템 프롬프트 스냅샷 포맷팅 |
| **Part 3** | **Episodic Memory (세션 기억)** | 3-1) 과거 세션 메시지 저장 & FTS5 인덱싱<br>3-2) [비교 실습] LLM 미사용 vs LLM 사용 요약/검색<br>3-3) 유저 질문 기반 1단계 요약 힌트 검색<br>3-4) Anchor 기반 2단계 대화 원문 정밀 인출 |
| **Part 4** | **MemoryMiddleware와 L4 주입** | 4-1) 고응집 도구 목록(`memory`, `session_recall`) 확인<br>4-2) `before_agent` 훅의 L4 주입 실물 덤프 |
| **Part 5** | **Context Compaction (컨텍스트 압축)** | 5-1) 도구로 입력되는 대형 컨텍스트 디스크 Swap<br>5-2) Swap 파일 부분 조회 (`grep_search`, `file_read`)<br>5-3) SummarizationMiddleware로 오래된 대화 요약 & 요약 프롬프트 바꾸기<br>5-4) SummarizationMiddleware 주요 파라미터<br>5-5) AI ↔ Tool 메시지 쌍 보존 |
| **Part 6** | **`create_agent`로 통합 에이전트 조립 & 시나리오** | 6-1) Prompt + Memory + Compaction 결합 에이전트 생성 (`user_id="notebook_temp_user"`)<br>6-2~6-4) 메모리 시나리오 3종<br>6-5) 대형 컨텍스트 Swap 시나리오<br>6-6) 대화 누적 → 자동 요약 → 이전 사실 기억 검증<br>6-7) 유저별 컨텍스트 격리 확인 (`artifacts/users/`) |
| **Part 7** | **[학습 정리] 프로덕션 운영 FAQ** | 메모리 운영, 압축 설정값, 요약 실패 처리, 가벼운 대안 |
| **Part 8** | **🧹 Clean-up & Reset** | 샌드박스 디렉토리, DB 커넥션, 실습 유저 디렉토리(`artifacts/users/notebook_*`) 정리 |

---

## 🛠️ Step 0. 환경 세팅 & 격리 샌드박스(Sandbox) 초기화

주피터 환경에서 비동기 루프를 안전하게 실행하기 위해 `nest_asyncio`를 활성화하고, **실습 전용 독립 임시 디렉토리(`demo_dir`)**와 **도구 결과 Swap 디렉토리(`demo_swaps_dir`)**를 생성합니다.

**`§` (Section Sign) 구분자로 명확히 나뉜 영문 마이크로 엔트리 구조**로 `USER.md`와 `MEMORY.md`를 직접 생성하여 독립적인 실습 환경을 구성합니다.

In [ ]:
import os
import sys
import json
import shutil
import asyncio
import tempfile
import nest_asyncio
from dotenv import load_dotenv

# 1. 비동기 루프 패치 (Jupyter 환경 필수)
nest_asyncio.apply()

# 2. 프로젝트 루트 상향 동적 탐색 (어느 서브 폴더에 노트북이 있어도 100% 작동)
def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), ".."))

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# 3. 환경 변수 로드
dotenv_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path)

# 4. 실습 격리용 샌드박스 디렉토리 생성
demo_dir = tempfile.mkdtemp(prefix="context_engineering_lab_")
demo_mem_dir = os.path.join(demo_dir, "database")
demo_epi_path = os.path.join(demo_dir, "episodic.db")
demo_chk_path = os.path.join(demo_dir, "checkpoints.db")
demo_swaps_dir = os.path.join(demo_dir, "tool-results")   # Part 5: 큰 도구 결과 Swap 저장소
os.makedirs(demo_mem_dir, exist_ok=True)

# 5. § 구분자 기반 표준 Semantic Memory 템플릿 생성 (영문)
demo_user_content = """- **Identity**: Cheolsu Kim (AI Software Engineer, 5 years experience)
§
- **Specialty**: AI Agent Engineering, LangChain, LangGraph, Distributed Software Architecture
§
- **Communication Style**: Professional and friendly; strongly prefers explanations focused on core code snippets and architectural diagrams
§
- **Hobbies & Interests**: Golf, weight training, cycling, swimming
§
- **Development Environment**: Primary IDE is VS Code; always provide technical explanations in Korean"""

demo_memory_content = """- **System Architecture**: 5-layer prompt stack + Hermes multi-layered memory (Semantic & Episodic) harness
§
- **Runtime Environment**: Python 3.12 (WSL Ubuntu) + FastAPI backend + Chainlit UI
§
- **Code Quality Policy**: Always verify generated code snippets for linting and execution validity before presenting them to the user"""

with open(os.path.join(demo_mem_dir, "USER.md"), "w", encoding="utf-8") as f:
    f.write(demo_user_content.strip() + "\n")

with open(os.path.join(demo_mem_dir, "MEMORY.md"), "w", encoding="utf-8") as f:
    f.write(demo_memory_content.strip() + "\n")

# 6. 노트북 하위 잔여 폴더 자동 정리
current_dir = os.getcwd()
for stale_dir in ["app", "artifacts"]:
    stale_path = os.path.join(current_dir, stale_dir)
    if os.path.exists(stale_path):
        shutil.rmtree(stale_path, ignore_errors=True)

print(f"✅ 환경 설정 및 격리 샌드박스 생성 완료!")
print(f"  - Project Root : {project_root}")
print(f"  - 실습용 Sandbox: {demo_dir}")
print(f"  - Swap 저장소   : {demo_swaps_dir}")
print(f"  - § 구분자 엔트리 생성: USER.md (5 entries), MEMORY.md (3 entries)")

## 📐 Part 1. 5-Layer Prompt Stack

상용 에이전트들은 프롬프트 캐싱과 관리의 효율성을 위해 프롬프트를 아래와 같이 계층 구조로 설계합니다.

```
┌─────────────────────────────────────────────────────────────────┐
│ Layer 1: System Identity & Core Role (PROMPT.md)                │  <-- [STATIC PREFIX: GPU KV-Cache HIT 🎯]
│ Layer 2: Capabilities (Tools & Skills)                          │
│   ├── 🛠️ Layer 2.1: Tool Capabilities (알파벳 정렬)             │
│   └── 📦 Layer 2.2: Available Skills Catalog (Frontmatter 스캔) │
│ ⚡ __SYSTEM_PROMPT_DYNAMIC_BOUNDARY__ (Cache Control 마킹 지점) │
├─────────────────────────────────────────────────────────────────┤
│ Layer 3: Dynamic Session Context (CWD, Session ID 등)           │  <-- [DYNAMIC SUFFIX: 가변 영역 ⚡]
│ Layer 4: Memory (Semantic: USER.md, MEMORY.md / Episodic 힌트)   │
│ Layer 5: User & Local Project Rules (AGENT.md)                  │
└─────────────────────────────────────────────────────────────────┘
```

### 1-1. 도구 및 스킬 카탈로그 빌더 준비

In [ ]:
from app.middleware.prompt import PromptAssembler, SkillPromptBuilder
from app.middleware.memory import SemanticMemoryStore, EpisodicStore, MemoryMiddleware
from app.tools import tools_chatbot

# 1. 실제 프로덕션 도구 로드 (기본 챗봇 도구 10개 + 메모리 미들웨어 도구 2개 = 총 12개)
epi_store = EpisodicStore(db_path=demo_epi_path)
await epi_store.setup()
sem_store = SemanticMemoryStore(memory_dir=demo_mem_dir)
sem_store.load_from_disk()

memory_mw = MemoryMiddleware(semantic_store=sem_store, episodic_store=epi_store)
all_real_tools = list(tools_chatbot) + memory_mw.get_tools()
print(f"🛠️ 등록된 실제 프로덕션 도구 수: {len(all_real_tools)}개")
print(f"  - 도구 목록: {[t.name for t in all_real_tools]}")

# 2. 스킬 디렉토리 스캔 및 Frontmatter 카탈로그 빌더 구성
skill_builder = SkillPromptBuilder(
    skills_dirs=[
        os.path.join(project_root, "skills"),
        os.path.join(project_root, ".agents/skills"),
    ],
    guidelines_path=os.path.join(project_root, "app/prompts/SKILL.md")
)
print(f"📦 SkillPromptBuilder 준비 완료")

### 1-2. 5-Layer 시스템 프롬프트 조립 & 덤프 출력

`PromptAssembler`는 **GPU KV-Cache 효율 극대화**를 위해 불변하는 정적 프리픽스(L1~L2)와 매 세션/턴마다 달라지는 동적 서픽스(L3~L5)를 명확히 분리하여 조립합니다.

| 계층 | 캐시 구간 | 계층 명칭 | 주요 포함 내용 및 주입 방식 |
|:---:|:---:|:---|:---|
| **L1** | **Static Prefix**<br>*(KV-Cache HIT 🎯)* | **System Identity & Core Role** | 에이전트의 기본 정체성, 핵심 원칙 및 헌법 (`system_rules` / `PROMPT.md`) |
| **L2** | **Static Prefix**<br>*(KV-Cache HIT 🎯)* | **Capabilities (Tools & Skills)** | • **L2.1 Tool Capabilities**: 등록된 도구 스키마를 이름 알파벳순 정렬 (`tool_schemas`)<br>• **L2.2 Skills Catalog**: `SKILL.md` 실행 지침 + 스킬 Frontmatter 요약표 (`skill_catalog`) |
| **⚡** | **Boundary** | **`__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__`** | 정적 프리픽스와 동적 서픽스를 가르는 프롬프트 캐시 컨트롤(`cache_control: ephemeral`) 경계선 |
| **L3** | **Dynamic Suffix**<br>*(세션/턴별 가변)* | **Dynamic Session Context** | 현재 날짜, 작업 디렉토리(`cwd`), `user_id`, `session_id`, 권한, **세션 격리 워크스페이스 경로(`workspace_dir`: `plan/`, `scripts/`, `outputs/`, `logs/`)** |
| **L4** | **Dynamic Suffix**<br>*(세션/턴별 가변)* | **Memory (Semantic & Episodic)** | • **Semantic Memory**: 세션 시작 시 고정된 장기 기억 스냅샷 (`MEMORY.md`, `USER.md`)<br>• **Episodic Memory**: 유저 질문 기반으로 FTS5에서 검색된 과거 세션 요약 힌트 (`MemoryMiddleware`가 `recalled_memory`로 통합 주입) |
| **L5** | **Dynamic Suffix**<br>*(세션/턴별 가변)* | **User & Local Project Rules** | `l5_docs`(또는 `agent_rules_path`)로 주입되는 프로젝트 로컬 규칙 및 컨벤션 문서 (`AGENT.md`, `CLAUDE.md`, `CONVENTIONS.md` 등) |

In [ ]:
from app.utils.workspace import resolve_session_workspace

# 1. Semantic Memory 스냅샷 전문 준비 (L4에 주입될 메모리 데이터)
recalled_memory_parts = []
mem_block = sem_store.format_for_prompt("memory")
usr_block = sem_store.format_for_prompt("user")
if mem_block:
    recalled_memory_parts.append(mem_block)
if usr_block:
    recalled_memory_parts.append(usr_block)
real_recalled_memory = "\n\n".join(recalled_memory_parts)

# 2. 5계층 프롬프트 조립기(PromptAssembler) 초기화
assembler = PromptAssembler(
    system_rules="You are an advanced AI Software Engineer equipped with long-term memory and precise reasoning capabilities.",
    tool_schemas=all_real_tools,            # L2.1: 실제 12개 도구 객체 주입 (알파벳 순 자동 정렬)
    skill_catalog=skill_builder.assemble,   # L2.2: Callable을 전달하여 동적 Frontmatter 카탈로그 생성
    l5_docs={                               # L5: 로컬 프로젝트 규칙 문서들 (텍스트 또는 파일 경로 지정 가능)
        "AGENT.md": "Always provide clear, structured, and verified responses.",
        "CONVENTIONS.md": "Use Python 3.12 type hints, async/await patterns, and save all artifacts inside the session workspace.",
    },
)

# 3. 세션 컨텍스트 (L3 & L4 가변 데이터 — 실제 PromptAssemblerMiddleware._prepare_messages와 동일 구조)
ws = resolve_session_workspace(
    user_id="cheolsu_kim",
    session_id="demo_lecture_session_001",
    ensure_exists=False,
)
session_context = {
    "cwd": project_root,
    "user_id": "cheolsu_kim",
    "session_id": "demo_lecture_session_001",
    "os": os.name,
    "workspace_dir": ws.rel_root,           # L3: 세션 격리 워크스페이스 루트 및 하위 폴더(plan, scripts, outputs, logs)
    "plan_dir": ws.rel_plan_dir,
    "scripts_dir": ws.rel_scripts_dir,
    "outputs_dir": ws.rel_outputs_dir,
    "logs_dir": ws.rel_logs_dir,
    "recalled_memory": real_recalled_memory # L4: MemoryMiddleware가 주입하는 장기/세션 기억 스냅샷
}

# 4. 5계층 완성 프롬프트 출력
full_prompt = assembler.build_system_prompt(session_context)
print("=" * 75)
print("📜 [5-Layer 조립된 시스템 프롬프트 출력 (L1 ~ L5 전문)]")
print("=" * 75)
print(full_prompt)
print("=" * 75)

await epi_store.close()

## 💾 Part 2. Semantic Memory (장기 기억: `MEMORY.md` & `USER.md`) 실습

Semantic Memory는 **에이전트의 관찰 노트(`MEMORY.md`)** 와 **사용자 프로필(`USER.md`)** 을 관리합니다.

### 🌟 핵심 설계 원칙
1. **`§` (Section Sign) 구분자**: 각 속성(Identity, Specialty, Hobbies 등)이 독립된 엔트리로 저장되어 세밀한 부분 수정이 가능합니다.
2. **글자 수(Character Count) 용량 제한**: 무한정 메모리가 증가하여 컨텍스트 윈도우를 고갈시키는 문제를 방지합니다.
3. **엔트리 단위 교체 (`replace`)**: `replace(target, old_text, new_content)`는 `old_text`를 포함하는 **해당 엔트리만 찾아 완성된 `new_content`로 독립 교체**합니다.
4. **Frozen Snapshot 패턴**: `load_from_disk()` 시점에 캡처된 스냅샷은 세션 중 도구로 엔트리를 추가해도 **현재 세션의 시스템 프롬프트에는 불변**으로 유지됩니다. (다음 세션부터 반영 ➔ 캐시 보존)

### 2-1. `§` 구분자로 분리된 초기 프로필 엔트리 확인

In [ ]:
store = SemanticMemoryStore(memory_dir=demo_mem_dir, memory_char_limit=4000, user_char_limit=2000)
store.load_from_disk()

print(f"📁 [초기 로드된 사용자 프로필 엔트리: 총 {len(store.user_entries)}개]:")
for idx, entry in enumerate(store.user_entries, 1):
    print(f"  [{idx}] {entry}")

### 2-2. 새 엔트리 추가(`add`) 및 중복 방지 테스트

메모리에 이미 있는 내용은 추가해도 저장되지 않음.

In [ ]:
# 1. 새 엔트리 추가 (새로운 독립 속성 등록)
r_add = store.add("user", "- **Preferred Diagrams**: Mermaid sequence diagrams and ASCII architecture diagrams")
print(f"➕ [추가 결과]: {r_add['message']} (현재 총 {r_add['entry_count']}개 엔트리, 사용량: {r_add['usage']})")

# 2. 동일 엔트리 중복 추가 시도
r_dup = store.add("user", "- **Preferred Diagrams**: Mermaid sequence diagrams and ASCII architecture diagrams")
print(f"🚫 [중복 추가 방지 결과]: {r_dup['message']}")

### 2-3. 특정 엔트리 단위 교체 (`replace`)

`§`로 각 속성이 분리되어 있으므로, **다른 인적사항(Identity, Specialty)에는 전혀 영향을 주지 않고 'Hobbies' 엔트리만 정확하게 수정**할 수 있습니다.

In [ ]:
# 'Hobbies' 키워드로 기존 취미 엔트리만 찾아 완성된 새 문장으로 교체
r_rep = store.replace(
    "user",
    "Hobbies",
    "- **Hobbies & Interests**: Tennis, CrossFit, Road biking, Triathlon"
)
print(f"✏️ [수정 결과]: {r_rep['message']}")
print("\n📁 [수정 후 전체 엔트리 목록]:")
for idx, entry in enumerate(store.user_entries, 1):
    print(f"  [{idx}] {entry}")

### 2-4. 시스템 프롬프트용 스냅샷 렌더링 확인

Part 1의 Prompt assembler에 이와 같은 USER, MEMORY 파일의 내용을 주입하는 기능이 들어있습니다.

In [ ]:
store.load_from_disk()  # 스냅샷 갱신
prompt_block = store.format_for_prompt("user")
print("📄 [시스템 프롬프트 주입용 블록 렌더링 결과 전문]:")
print("─" * 60)
print(prompt_block)
print("─" * 60)

## 📚 Part 3. Episodic Memory (세션 기억: SQLite + FTS5) & 2-Stage JIT 회상 실습

에피소드 메모리는 과거 세션의 대화를 SQLite에 저장하고, **FTS5(Full-Text Search)**를 통해 의미 있는 과거 대화를 검색합니다.

### 🌟 2-Stage JIT (Just-In-Time) 회상 흐름
1. **`finalize_session()`**: 세션 종료 시 메시지 저장 + 요약(Summary) & 키워드 생성 + FTS5 인덱싱.
2. **1단계 (자동 힌트 주입)**: `search_sessions(query)`로 현재 유저 query와 관련된 과거 세션에 대한 요약을 프롬프트에 주입.
3. **2단계 (Anchor 기반 정밀 인출)**: `get_anchored_view(session_id, anchor_keyword)`로 특정 메시지 중심 ±window 범위 및 첫/끝 bookend 메시지 인출.

> 💡 **세션 데이터 이원화 (`transcript.db` + `episodic.db`)**: 실시간 대화 원문(질문·답변·도구 호출·결과)은 `TranscriptStore`(`transcript.db`)에 Append-only로 즉시 백업되고, `EpisodicStore`(`episodic.db`)에는 다중 주제(Topics) 요약과 키워드 색인만 저장됩니다.

### 3-1. 과거 세션 메시지 저장 & FTS5 인덱싱

세션 메시지들을 인덱싱하는 과정을 보여줍니다. 일반적으로 세션이 종료됐다고 판단될 때 대화 내용을 요약해서 인덱싱합니다.

In [ ]:
es = EpisodicStore(db_path=demo_epi_path)
await es.setup()

session_a_msgs = [
    {"role": "human", "content": "도커 컨테이너 빌드 시 캐시 최적화는 어떻게 하나요?"},
    {"role": "ai", "content": "requirements.txt를 먼저 COPY하여 pip install을 수행하고, 소스코드는 나중에 COPY합니다."},
    {"role": "human", "content": "멀티스테이지 빌드는 어떻게 작성하죠?"},
    {"role": "ai", "content": "builder 스테이지에서 빌드 아티팩트를 생성하고, 최종 runner 이미지에는 빌드 도구 없이 최소 런타임만 포함합니다."},
    {"role": "human", "content": "도커 컴포즈에서 restart policy 추천은?"},
    {"role": "ai", "content": "프로덕션 서비스는 restart: always 또는 unless-stopped 설정을 권장합니다."}
]
await es.finalize_session("session_docker_opt_001", session_a_msgs, llm=None)
print("💾 [1. 세션 A 대화 6개 메시지 저장 및 FTS5 인덱싱 완료]")

### 3-2. Episodic session 정보 인덱싱 아키텍처: 1단계(로컬) => 2단계(LLM 사용)

1. **1단계 (로컬 규칙 기반)**: 대화 원문에 실제로 등장한 한/영 단어를 불용어 필터링 후 추출 → FTS5 색인
2. **2단계 (LLM 지능형 보완)**: LLM이 대화 전체를 읽고 핵심 결론 요약 + 원문에 없는 상위 개념·동의어·한영 매핑 키워드 추출 → 1단계와 병합

**💡 핵심 설계 원칙 — 원문 단어 손실 0%:**
- LLM이 키워드를 영어로만 번역하거나, 유의어로 치환해 버려도, **1단계에서 유저가 실제로 말한 한글 원문 단어("쿠키", "보안", "토큰" 등)가 키워드 세트에 100% 보존**됩니다.
- 따라서 한국어로 검색해도, 영어로 검색해도 **검색 누락이 차단**됩니다!

**⚡ 프로덕션 백그라운드 실행:**
- 실제 프로덕션에서는 `MemoryMiddleware.after_agent`가 **`IdleReviewQueue`(기본 유휴 15초 / 최대 30분)**에 백그라운드 리뷰를 예약하여, 대화가 멈춘 뒤 `transcript.db` 전체 원문을 읽고 `finalize_session(llm=review_llm)`을 실행합니다 (그사이 사용자가 새 질문을 입력하면 대기 중인 리뷰는 즉시 취소).

In [ ]:
from app.utils import init_chat_model

# 1. 요약 테스트용 LLM 초기화 (configs/model.config)
with open(os.path.join(project_root, "configs/model.config"), "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
summary_llm = init_chat_model(
    model=model_cfg.get("model_name", "gemini-3.8-flash"),
    temperature=0.0
)

# 2. 대화 후반부에 핵심 기술 결정(쿠키 보안)이 나오는 대화 데이터셋
compare_msgs = [
    {"role": "human", "content": "오늘 마이크로서비스 인증 아키텍처 회의를 시작합시다."},
    {"role": "ai", "content": "네, 세션 기반 인증과 JWT 토큰 인증 중 어떤 방식을 검토할까요?"},
    {"role": "human", "content": "확장성을 위해 Stateless JWT 방식으로 진행합시다."},
    {"role": "ai", "content": "좋습니다. Stateless JWT 아키텍처로 진행하겠습니다."},
    {"role": "human", "content": "Refresh Token 보안 저장소 위치는 어디로 확정할까요?"},
    {"role": "ai", "content": "XSS 공격 방지를 위해 클라이언트 로컬스토리지 대신 HTTP-Only Secure Cookie에 보관합니다."}
]

# A. LLM 미사용 (1단계 로컬 키워드 추출만 적용)
await es.finalize_session("session_compare_rule_based", compare_msgs, llm=None)

# B. LLM 사용 (1단계 로컬 + 2단계 LLM 지능형 키워드 병합)
await es.finalize_session("session_compare_llm_based", compare_msgs, llm=summary_llm)

# 3. DB에 저장된 실제 요약(Summary) 및 키워드(Keywords) 1:1 대조 출력
rows = await es._conn.execute_fetchall(
    "SELECT session_id, summary, keywords FROM sessions WHERE session_id IN (?, ?)",
    ("session_compare_rule_based", "session_compare_llm_based")
)
print("=" * 75)
print("📊 [1. Progressive Keyword Pipeline 결과 비교]")
print("=" * 75)
for sid, s_summary, s_keywords in rows:
    if sid == "session_compare_rule_based":
        print(f"\n❌ [LLM 미사용 (1단계: 로컬 규칙 기반 키워드만 적용)]:")
        print(f"   - Summary : {s_summary}")
        print(f"   - Keywords: {s_keywords}")
    elif sid == "session_compare_llm_based":
        print(f"\n🎯 [LLM 사용 (1단계 로컬 + 2단계 LLM 한/영 키워드 병합)]:")
        print(f"   - Summary : {s_summary}")
        print(f"   - Keywords: {s_keywords}")

In [ ]:
# 4. 핵심 비교: '쿠키 보안' 한국어 FTS5 검색 결과
print("\n" + "=" * 75)
print("🔍 [2. '쿠키 보안' 한국어 검색 — 원문 키워드 보존 효과 검증]")
print("=" * 75)
search_cookie = await es.search_sessions("쿠키 보안", top_k=5)
cookie_sids = [(s["session_id"], s['summary']) for s in search_cookie]

print(f"🔎 한국어 검색어: '쿠키 보안' ➔ 검색된 세션: {cookie_sids}")

if not cookie_sids:
    print("  ⚠️ 검색 결과 없음 — 키워드 인덱스를 확인하세요.")

In [ ]:
# 5. 영문 기술어 검색 — LLM 보완의 부가가치 확인
print("\n" + "=" * 75)
print("🔍 [3. 'Stateless Authentication' 영문 기술어 검색 — LLM 보완 효과 검증]")
print("=" * 75)
search_auth = await es.search_sessions("Stateless Authentication", top_k=5)
auth_sids = [(s["session_id"], s['summary']) for s in search_auth]

print(auth_sids)

### 3-3. 유저 질문 기반 FTS5 검색 (`search_sessions`)

관련 세션을 검색해보는 1차 검색으로 차후 프롬프트에 Epidodic Memory로 자동 주입될 정보. 이 정보는 관련한 대화에 대한 기억을 에이전트가 자동으로 떠올리는 것에 비유할 수 있음.

In [ ]:
search_results = await es.search_sessions("도커 멀티스테이지 빌드", top_k=2)
print("🔍 [FTS5 검색 결과 (1단계 검색)]:")
for res in search_results:
    print(f"  - Session ID: {res['session_id']}")
    print(f"    Summary   : {res['summary']}")
    print(f"    Keywords  : {res['keywords']}")

### 3-4. Anchor 기반 2단계 대화 인출 (`session_recall`)

Session ID와 keyword로 관련한 대화 원문을 추출합니다. 에이전트의 도구를 통해 주입될 정보. 즉 에이전트가 1차 검색을 토대로 필요하다고 생각할 때, 기억을 더듬어 관련 대화 기록을 불러오는 기능.

In [ ]:
anchored_msgs = await es.get_anchored_view(
    session_id="session_docker_opt_001",
    anchor_keyword="멀티스테이지",
    window=2
)
print(f"🎯 [Anchor('멀티스테이지') 기반 정밀 인출 ({len(anchored_msgs)}개 메시지 전문)]:")
for idx, m in enumerate(anchored_msgs, 1):
    print(f"  [{idx}] {m['role'].upper()}: {m['content']}")

await es.close()

## 🔌 Part 4. MemoryMiddleware와 고응집 도구 바인딩 & L4 주입 검증

미들웨어가 메모리 스토어뿐 아니라 에이전트가 사용할 **도구(`memory`, `session_recall`)까지 소유**하는 고응집(High-Cohesion) 구조입니다.

### 🌟 `before_agent` 훅을 통한 L4 메모리 단일 주입 과정
1. `semantic_store.format_for_prompt()` ➔ **Semantic Frozen Snapshot**
2. `episodic_store.search_sessions(user_query)` ➔ **Episodic FTS5 요약 힌트**
3. 위 두 가지를 `ctx.recalled_memory`에 결합하여 L4에 단일 주입 (이중 주입 방지)

### 4-1. 메모리 도구 목록 확인

Memory Middleware를 통해 자동 주입되는 메모리와 관련한 도구들.

In [ ]:
sem_store = SemanticMemoryStore(memory_dir=demo_mem_dir)
sem_store.load_from_disk()
epi_store = EpisodicStore(db_path=demo_epi_path)
await epi_store.setup()

mw = MemoryMiddleware(semantic_store=sem_store, episodic_store=epi_store)

tools = mw.get_tools()
print(f"🛠️ MemoryMiddleware가 제공하는 도구 목록 ({len(tools)}개):")
for t in tools:
    print(f"  - 이름: {t.name:<15} 설명: {t.description}...")

### 4-2. `before_agent` 훅 실행 및 L4 주입 데이터 덤프 검증

Memory Middleware는 before_agent 단계에서 runtime의 recalled_memory 변수에 기억 주입. 차후 Prompt assembler Middleware에 의해 프롬프트에 주입.

In [ ]:
from langchain_core.messages import HumanMessage
from app.context import AgentContext

ctx = AgentContext(semantic_memory_enabled=True, episodic_memory_enabled=True)
class DummyRuntime:
    def __init__(self, ctx):
        self.context = ctx
        self.config = {"configurable": {"thread_id": "demo_thread_001"}}

fake_state = {"messages": [HumanMessage(content="도커 빌드 캐시 최적화에 대해 질문할게요.")]}
mw.before_agent(fake_state, DummyRuntime(ctx))

print("📥 [before_agent 훅에 의해 ctx.recalled_memory에 자동 주입된 전체 내용 (전문)]:")
print("─" * 75)
print(ctx.recalled_memory)
print("─" * 75)

await epi_store.close()

## 🗜️ Part 5. Context Compaction (컨텍스트 압축)

Part 1~4는 **무엇을** 컨텍스트에 넣고 남길지(Select, Write)를 다뤘습니다. Part 5는 그 컨텍스트가 **넘치지 않게 줄이는 방법**(Compress)입니다.

### 💡 컨텍스트가 넘치는 두 가지 경로와 해법
1. **도구로 입력되는 대형 컨텍스트**: 웹 문서·로그·대형 표 하나가 수만 자. 몇 번만 호출해도 컨텍스트가 찹니다.
   ➔ **`ToolResultSwapMiddleware`**: 도구가 끝나는 즉시 결과를 **디스크에 저장**하고, 모델에게는 **미리보기 + 파일 경로**만 보여줍니다. 필요한 부분은 에이전트가 `grep_search` / `file_read`로 다시 찾아 읽습니다.
2. **대화가 쌓임**: 턴이 늘어날수록 이전 질문·답변·도구 기록이 누적됩니다.
   ➔ **`SummarizationMiddleware` (LangChain prebuilt)**: 대화 토큰이 `trigger`를 넘으면, 최근 `keep` 분량만 원문으로 남기고 **앞부분을 LLM으로 요약**해 교체합니다.

```mermaid
flowchart LR
    Q["사용자 질문"] --> BM{"before_model<br>대화 토큰 ≥ trigger?"}
    BM -- "Yes" --> S["SummarizationMiddleware<br>앞부분 요약 + 최근 keep 유지"]
    S --> M["LLM 호출"]
    BM -- "No" --> M
    M -- "tool_calls" --> T["도구 실행"]
    T --> W{"wrap_tool_call<br>결과 > 10,000자?"}
    W -- "Yes" --> D["디스크 저장<br>미리보기 + 경로만 대화에"]
    W -- "No" --> R["결과 그대로 대화에"]
    D --> BM
    R --> BM
```

| 미들웨어 | 훅 | 대상 | 방식 | 출처 |
|:---|:---|:---|:---|:---|
| `ToolResultSwapMiddleware` | `wrap_tool_call` | 10,000자 넘는 도구 결과 1건 | 디스크 저장 + 2,000자 미리보기 | 하네스 구현 (약 100줄) |
| `SummarizationMiddleware` | `before_model` | 누적된 대화 전체 | LLM 요약으로 state 교체 | **LangChain prebuilt** |

> 💡 **왜 prebuilt를 쓰나요?** 요약은 "어디서 자를지(AI ↔ Tool 쌍 보존)", "실패하면 어떻게 할지", "요약 프롬프트를 어떻게 구성할지" 같은 함정이 많습니다. 검증된 prebuilt를 쓰고, 하네스에 필요한 부분만 얇게 덧붙이는 것이 안전합니다.

### 5-1. `ToolResultSwapMiddleware` — 도구로 입력되는 대형 컨텍스트 디스크 Swap

- **언제**: 도구가 결과를 돌려주는 즉시(`wrap_tool_call`) **한 번만** 처리합니다. 대화(state)에는 처음부터 미리보기가 저장되므로 매 모델 호출마다 다시 처리하지 않습니다.
- **어떻게**: 원문은 파일(`<tool_call_id>.txt`)로 저장하고, 모델에게는 `<persisted-output>` 블록(크기, 파일 경로, 미리보기)을 보여줍니다.
- **어디에**: `swap_dir`를 주면 `<swap_dir>/<thread_id>/`에, 주지 않으면 **유저·세션 워크스페이스** `artifacts/users/{user_id}/sessions/{session_id}/tool-results/`에 저장합니다 (Part 6에서 확인). 이 셀은 실습 샌드박스를 `swap_dir`로 지정합니다.
- 아래 셀은 200개 섹션짜리 대형 웹 문서(120번 섹션에 핵심 경고 포함)를 Swap해 봅니다.

In [ ]:
import re
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from app.middleware.compaction import ToolResultSwapMiddleware

swap_mw = ToolResultSwapMiddleware(swap_dir=demo_swaps_dir, max_result_chars=10_000, preview_chars=2_000)

# 1. 약 2만 자짜리 대형 웹 문서 (Section 120에 핵심 보안 경고 포함)
large_web_doc = "<h1>OpenID Connect & OAuth 2.0 Security Best Practices 2026</h1>\n"
for i in range(1, 201):
    if i == 120:
        large_web_doc += f"<section id='vuln'>Section {i}: CRITICAL WARNING - Always validate PKCE code_verifier with SHA-256!</section>\n"
    else:
        large_web_doc += f"<section>Section {i}: Standard token exchange protocols and identity federation metadata details...</section>\n"

raw_msg = ToolMessage(content=large_web_doc, tool_call_id="call_oauth_web", name="web_fetch")
print(f"📄 원본 도구 결과: {len(large_web_doc):,}자 (약 {len(large_web_doc)//4:,} 토큰)")

# 2. Swap 적용 (create_agent 안에서는 wrap_tool_call이 이 메서드를 자동 호출)
swapped_msg = swap_mw.swap_if_large(raw_msg)
print(f"💾 Swap 후 대화에 남는 크기: {len(swapped_msg.content):,}자 (절감률 {(1 - len(swapped_msg.content)/len(large_web_doc))*100:.1f}%)")
print("=" * 80)
print(swapped_msg.content[:700])
print("   ... (미리보기 중략) ...")
print(swapped_msg.content[-120:])
print("=" * 80)

#### 동작 규칙 확인: 중복 저장 방지 & 제외 대상
- 같은 결과(같은 `tool_call_id`)를 다시 처리해도 **같은 파일**을 가리키고, 이미 Swap된 메시지는 건드리지 않습니다.
- 작은 결과는 그대로 둡니다.
- **`file_read` 결과는 Swap하지 않습니다.** Swap 파일을 `file_read`로 다시 읽었는데 그 결과가 또 Swap되면, 에이전트는 영원히 내용을 볼 수 없는 순환에 빠집니다. `file_read`는 `offset`/`limit`으로 스스로 분량을 제한하는 도구라 제외 대상이 됩니다.

In [ ]:
# 1. 같은 결과를 다시 처리 → 같은 파일, 같은 미리보기
again = swap_mw.swap_if_large(raw_msg)
print(f"🔄 재처리 결과가 동일한가?          : {again.content == swapped_msg.content}")
print(f"🔄 이미 Swap된 메시지는 그대로인가? : {swap_mw.swap_if_large(swapped_msg) is swapped_msg}")

# 2. 작은 결과는 그대로
small_msg = ToolMessage(content="Found 3 matching files.", tool_call_id="call_small", name="grep_search")
print(f"📎 작은 결과는 그대로인가?          : {swap_mw.swap_if_large(small_msg) is small_msg}")

# 3. file_read 결과는 아무리 커도 Swap하지 않음 (재Swap 순환 방지)
read_msg = ToolMessage(content=large_web_doc, tool_call_id="call_read", name="file_read")
print(f"📖 file_read 결과는 그대로인가?     : {swap_mw.swap_if_large(read_msg) is read_msg}")

# 4. 디스크에 저장된 파일 확인 (그래프 밖에서 호출했으므로 'default' 세션 폴더)
for root, _, files in os.walk(demo_swaps_dir):
    for f in files:
        p = os.path.join(root, f)
        print(f"🗂️ 저장 파일: {os.path.relpath(p, demo_swaps_dir)} ({os.path.getsize(p):,} bytes)")

### 5-2. Swap 파일에서 필요한 부분만 다시 찾아 읽기

에이전트는 미리보기에 원하는 정보가 없으면, 안내된 경로에 대해 **`grep_search`로 위치를 찾고 `file_read`로 그 주변만** 읽습니다. 아래 셀은 실제 프로젝트 도구를 직접 호출해 그 과정을 재현합니다.

In [ ]:
from app.tools.common import grep_search, file_read

# 1. 미리보기 블록에서 파일 경로 추출
swap_file_path = re.search(r"Full output saved to: (\S+)", swapped_msg.content).group(1)
print(f"📍 Swap 파일 경로: {swap_file_path}\n")

# 2. grep_search로 핵심 키워드 위치 찾기
hit = grep_search.invoke({"pattern": "CRITICAL", "search_path": swap_file_path, "output_mode": "content"})
print(f"🔎 grep_search 결과:\n{hit}\n")

# 3. 찾은 줄 주변만 file_read로 읽기
line_no = int(re.search(r":(\d+):", hit).group(1))
snippet = file_read.invoke({"file_path": swap_file_path, "offset": line_no - 2, "limit": 5})
print(f"📖 file_read 결과 (줄 {line_no - 2}~{line_no + 2}):\n{snippet}")
print(f"✨ 약 {len(large_web_doc):,}자 전체 대신 {len(hit) + len(snippet):,}자만 컨텍스트에 들어갔습니다.")

### 5-3. `SummarizationMiddleware` — 오래된 대화를 요약으로 교체

LangChain의 `SummarizationMiddleware`는 `before_model` 훅에서 다음을 수행합니다.

1. 대화(`state["messages"]`)의 추정 토큰 또는 직전 모델 호출의 실사용 토큰(`usage_metadata`)이 **`trigger`** 이상인지 확인합니다.
2. 최근 **`keep`** 분량을 원문으로 남길 위치(cutoff)를 정합니다. 이때 `AIMessage(tool_calls)`와 `ToolMessage`가 갈라지지 않도록 위치를 조정합니다.
3. cutoff 앞부분을 요약 LLM에 보내 **SESSION INTENT / SUMMARY / ARTIFACTS / NEXT STEPS** 4개 섹션으로 요약합니다.
4. `RemoveMessage(REMOVE_ALL_MESSAGES)` + `[요약 메시지, 최근 메시지...]`를 반환해 **checkpointer state를 실제로 교체**합니다.
5. 요약 호출이 실패하면 3회 재시도 후 에러를 올립니다. 가짜 요약으로 대화를 덮어쓰지 않습니다.

하네스는 이를 상속한 **`HarnessSummarizationMiddleware`**로 세 가지만 덧붙입니다.

| 덧붙인 것 | 이유 |
|:---|:---|
| 요약 메시지에 `harness_origin` 표식과 머리말 | 요약은 `HumanMessage`로 들어갑니다. 표식이 없으면 StopHooks·Memory가 이를 **새 사용자 턴**으로 셉니다. |
| 요약 호출에 `exclude_from_stream` 태그 | 서버가 요약 LLM의 토큰을 **답변으로 사용자에게 스트리밍**하지 않도록 합니다. |
| `transcript_store` 압축 이벤트 기록 | 원문 전체는 `transcript.db`에 남고, 언제 무엇이 요약됐는지도 기록합니다. |

아래 셀은 JWT 인증 설계 대화(파일 읽기 도구 호출 포함)를 만들고, 실습을 위해 `trigger`를 아주 작게 잡아 요약을 일으켜 봅니다.

In [ ]:
from langchain_core.messages.utils import count_tokens_approximately
from app.middleware.compaction import HarnessSummarizationMiddleware
from app.utils.message_markers import is_user_turn_start, is_compaction_summary, user_turn_starts
from app.utils import init_chat_model

with open(os.path.join(project_root, "configs/model.config"), "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
llm = init_chat_model(model=model_cfg.get("model_name", "gemini-3.8-flash"), temperature=0.0)

# 1. 실습용 대화: 설계 결정 → 파일 읽기(도구) → 저장소 결정 → 현재 작업
auth_code = "\n".join(
    ["# auth_service.py", "import jwt", "from datetime import datetime, timedelta", "",
     "SECRET_KEY = 'demo-secret'", "",
     "def create_access_token(data: dict, minutes: int = 30):",
     "    payload = {**data, 'exp': datetime.utcnow() + timedelta(minutes=minutes)}",
     "    return jwt.encode(payload, SECRET_KEY, algorithm='HS256')", ""]
    + [f"def _policy_rule_{i:03d}(token): return True  # validation rule {i}" for i in range(1, 60)]
)

def tcall(name, args, call_id):
    return {"name": name, "args": args, "id": call_id, "type": "tool_call"}

long_history = [
    HumanMessage("FastAPI 서비스에 JWT 인증을 붙이려고 해. 설계부터 같이 하자.", id="u1"),
    AIMessage("세션 방식은 서버가 상태를 저장해야 해서 수평 확장이 어렵고, JWT는 토큰 자체에 클레임을 담아 "
              "어느 서버에서나 검증할 수 있습니다. 마이크로서비스 확장성을 고려하면 Stateless JWT를 권장합니다.", id="a1"),
    HumanMessage("Stateless JWT로 가자. Access/Refresh 토큰 유효기간은 어떻게 할까?", id="u2"),
    AIMessage("탈취 시 피해를 줄이기 위해 Access Token은 15분으로 짧게, 재로그인 부담을 줄이기 위해 "
              "Refresh Token은 7일을 제안합니다.", id="a2"),
    HumanMessage("좋아, 그걸로 확정. 현재 src/auth_service.py를 읽고 어디를 고쳐야 할지 알려줘.", id="u3"),
    AIMessage("", tool_calls=[tcall("file_read", {"file_path": "src/auth_service.py"}, "call_read_auth")], id="a3"),
    ToolMessage(auth_code, tool_call_id="call_read_auth", name="file_read", id="t3"),
    AIMessage("create_access_token의 기본 만료가 30분이라 15분으로 바꿔야 하고, Refresh Token 발급 함수가 아직 없습니다.", id="a3b"),
    HumanMessage("Refresh Token은 어디에 저장하지?", id="u4"),
    AIMessage("XSS로 탈취되지 않도록 HTTP-Only Secure Cookie에 저장하고, 로그아웃 시 토큰의 jti를 "
              "Redis 블랙리스트에 등록해 즉시 무효화합니다.", id="a4"),
    HumanMessage("좋아. 이제 create_access_token의 만료 로직부터 수정해보자.", id="u5"),
]
print(f"📚 원본 대화: {len(long_history)}개 메시지, 약 {count_tokens_approximately(long_history):,} 토큰")

# 2. 요약 미들웨어 (실습용으로 trigger/keep을 아주 작게 설정)
summ_mw = HarnessSummarizationMiddleware(
    llm,
    trigger=("tokens", 1200),        # 대화가 1,200 토큰 이상이면 요약
    keep=("tokens", 300),            # 최근 300 토큰 분량은 원문 유지
    trim_tokens_to_summarize=None,   # 요약 대상 전체를 요약 LLM에 전달 (5-4 참고)
)

# 3. before_model 훅 직접 실행 (create_agent 안에서는 매 모델 호출 전에 자동 실행)
update = summ_mw.before_model({"messages": long_history}, None)
new_messages = update["messages"][1:]   # [0]은 RemoveMessage(REMOVE_ALL_MESSAGES): 기존 state 전체 삭제 지시
print(f"🗜️ 요약 후: {len(new_messages)}개 메시지, 약 {count_tokens_approximately(new_messages):,} 토큰")
print("=" * 80)
for idx, m in enumerate(new_messages):
    tag = "📝 요약" if is_compaction_summary(m) else "📌 원문"
    preview = str(m.content)[:70].replace("\n", " ")
    print(f"[{idx}] {tag} {type(m).__name__:13s}: {preview}...")

In [ ]:
summary_msg = new_messages[0]
print("📝 [요약 메시지 전문]")
print("─" * 80)
print(summary_msg.content)
print("─" * 80)

# 요약 메시지는 HumanMessage지만 '사용자 턴'으로 세지 않음
print(f"요약 메시지가 사용자 턴 시작인가? : {is_user_turn_start(summary_msg)}")
print(f"사용자 턴 수 (요약 전 → 후)       : {len(user_turn_starts(long_history))} → {len(user_turn_starts(new_messages))}")

#### ✏️ 요약 프롬프트 바꾸기 (`summary_prompt`)

**무엇을, 어떤 형식으로 요약할지는 요약 프롬프트가 결정합니다.** 위 요약이 `SESSION INTENT / SUMMARY / ARTIFACTS / NEXT STEPS` 4개 섹션으로 나온 이유는, LangChain 기본 프롬프트(`DEFAULT_SUMMARY_PROMPT`)가 그 형식을 지시하기 때문입니다. 이 프롬프트는 `summary_prompt` 파라미터로 통째로 바꿀 수 있습니다.

- **규칙 1 — `{messages}` 자리표시자 필수**: 요약할 대화가 `summary_prompt.format(messages=...)`로 이 자리에 들어갑니다.
- **규칙 2 — 다른 중괄호는 두 번**: `str.format`을 쓰므로 JSON 예시 등 프롬프트 안의 `{ }`는 `{{ }}`로 써야 합니다.
- **활용 예**: 요약 언어 지정(한국어), 도메인별 필수 섹션 추가, "숫자·ID·파일 경로는 원문 그대로 남겨라" 같은 보존 규칙.

```python
KOREAN_SUMMARY_PROMPT = """당신은 에이전트의 대화 기록을 압축하는 도우미입니다.
아래 대화는 이 요약으로 대체되므로, 작업을 이어가는 데 필요한 정보만 빠짐없이 남기세요.
반드시 한국어로, 다음 섹션을 모두 채우세요 (없으면 "없음"):

## 사용자 목표
## 확정된 결정과 수치 (숫자·ID·파일 경로·코드명은 원문 그대로)
## 읽거나 수정한 파일과 산출물
## 남은 작업

<messages>
{messages}
</messages>"""

# 하네스 팩토리
compaction_mws = create_compaction_middleware(llm, trigger_tokens=..., summary_prompt=KOREAN_SUMMARY_PROMPT)
# 또는 미들웨어 직접 생성
summ_mw = HarnessSummarizationMiddleware(llm, trigger=("tokens", 1200), keep=("tokens", 300),
                                         summary_prompt=KOREAN_SUMMARY_PROMPT)
```

> 💡 기본 프롬프트를 조금만 고치고 싶다면 `from langchain.agents.middleware.summarization import DEFAULT_SUMMARY_PROMPT`로 원문을 가져와 일부를 바꿔 쓰면 됩니다. 예: `DEFAULT_SUMMARY_PROMPT.replace("<messages>", "Write the summary in Korean.\n\n<messages>")`

### 5-4. `SummarizationMiddleware` 주요 파라미터

| 파라미터 | 의미 | 지정 형식 (기본값) | `create_compaction_middleware` 설정 |
|:---|:---|:---|:---|
| `model` | 요약을 생성할 LLM | 모델 객체 또는 이름 | `summary_llm` (없으면 에이전트 `llm`) |
| `trigger` | **언제** 요약할지 | `("tokens", N)` / `("messages", N)` / `("fraction", 0.8)`<br>`{"tokens": N, "messages": M}` (AND), 리스트 (OR)<br>(기본값 `None` = 요약하지 않음) | `("tokens", trigger_tokens)` |
| `keep` | 요약 후 **원문 그대로 남길** 최근 분량 | `("tokens", N)` / `("messages", N)` / `("fraction", 0.2)`<br>(기본값 `("messages", 20)`) | `("tokens", keep_tokens)` |
| `summary_prompt` | **무엇을, 어떤 형식으로** 요약할지 | `{messages}`를 포함한 문자열 (기본값 `DEFAULT_SUMMARY_PROMPT`) | `summary_prompt` (없으면 기본값) |
| `trim_tokens_to_summarize` | 요약 LLM에 넘길 대화의 **최대 토큰**. 넘치면 **뒤쪽부터** 남기고 앞부분은 버림 | 정수 또는 `None` (기본값 `4000`) | `None`: 요약 대상 전체 전달 (요약 대상은 trigger 크기를 넘지 않음) |
| `token_counter` | 토큰 수를 세는 함수 | 함수 (기본값 문자 수 기반 근사) | 기본값 |

- `trigger`는 **대화 메시지 추정치**와 **직전 모델 호출의 실사용량**(`usage_metadata`, 시스템 프롬프트·도구 스키마 포함) 중 하나라도 넘으면 발동합니다 (Part 6에서 확인).
- `fraction` 지정은 모델 프로필에 `max_input_tokens`가 있어야 쓸 수 있습니다.

### 5-5. AI ↔ Tool 메시지 쌍 보존

`AIMessage(tool_calls)`와 그 결과 `ToolMessage`는 반드시 함께 있어야 합니다. 결과만 남고 호출이 사라지면(고아 ToolMessage) 대부분의 모델 API가 **400 에러**를 냅니다.

`SummarizationMiddleware`가 남길 구간의 시작점(cutoff)을 정하는 규칙은 다음과 같습니다.
1. 먼저 `keep` 기준으로 cutoff를 잡습니다. (예: `keep=("messages", 5)`면 뒤에서 5개)
2. cutoff 자리가 `ToolMessage`이면, **그 도구를 호출한 `AIMessage`를 뒤로 거슬러 찾아 cutoff를 그 앞으로 옮깁니다.**
3. 그래서 **요청보다 더 많이 남기고, 그만큼 덜 요약합니다.** 아래 예에서는 5개를 요청했지만 `AIMessage`가 함께 남아 **6개**가 남습니다.
   (짝 `AIMessage`를 찾지 못하는 예외 상황에서만, 반대로 그 `ToolMessage`들을 건너뛰어 더 적게 남깁니다.)

In [ ]:
keep_n = 5
naive_cut = len(long_history) - keep_n
safe_cut = summ_mw._find_safe_cutoff(long_history, keep_n)   # 내부 메서드: 쌍 보존을 반영한 cutoff

def describe(i):
    m = long_history[i]
    return f"[{i}] {type(m).__name__}" + (f" (tool_calls: {m.tool_calls[0]['name']})" if getattr(m, "tool_calls", None) else "")

print(f"요청: 최근 {keep_n}개 메시지 유지 (전체 {len(long_history)}개)\n")
print(f"✂️ 단순히 뒤에서 {keep_n}개 → {len(long_history) - naive_cut}개 유지, 첫 메시지 {describe(naive_cut)}  ⚠️ 호출 없는 고아 결과")
print(f"🛡️ 쌍 보존 cutoff     → {len(long_history) - safe_cut}개 유지, 첫 메시지 {describe(safe_cut)}  ✅ 호출과 결과가 함께 남음\n")
print("📌 실제로 원문 그대로 남는 메시지:")
for i in range(safe_cut, len(long_history)):
    print(f"   {describe(i)}: {str(long_history[i].content)[:50].replace(chr(10), ' ')}")

## 🚀 Part 6. `create_agent`로 통합 에이전트 조립 및 시나리오 테스트

Part 1~5의 **LLM, 5-Layer PromptAssembler, MemoryMiddleware, Compaction(Swap + Summarization), Checkpointer**를 `create_agent`로 **노트북 안에서 직접 조립**하고, 시나리오를 각각 독립 셀에서 실행합니다.

**미들웨어 순서**: `[memory_mw, *compaction_mws, prompt_mw]`
- `memory_mw.before_agent`가 L4 기억을 준비하고,
- `SummarizationMiddleware.before_model`이 대화를 먼저 줄인 뒤,
- `prompt_mw.wrap_model_call`이 5-Layer 시스템 프롬프트를 붙여 모델을 호출합니다.
- 도구 결과는 `ToolResultSwapMiddleware.wrap_tool_call`이 대화에 들어가기 전에 Swap합니다.

**실습 유저**: Part 6의 모든 시나리오는 `AgentContext(user_id="notebook_temp_user")`로 실행합니다. 장기 기억·세션 워크스페이스·Swap 파일이 실제 서버와 같은 경로인 `artifacts/users/notebook_temp_user/` 아래에 쌓이고, 6-7에서 다른 유저와 섞이지 않는지 확인합니다. (Part 8에서 삭제)

### 6-1. 모든 빌딩 블록을 결합하여 `create_agent`로 에이전트 인스턴스 빌드

In [ ]:
import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from app.utils import init_chat_model
from app.middleware.prompt import PromptAssembler, SkillPromptBuilder, create_prompt_assembler_middleware
from app.middleware.memory import SemanticMemoryStore, EpisodicStore, MemoryMiddleware
from app.tools import tools_chatbot
from app.context import AgentContext
from app.utils.message_utils import normalize_content
from app.middleware.compaction import create_compaction_middleware
from app.utils.message_markers import is_compaction_summary
from app.utils.workspace import USERS_ROOT_DIR, resolve_user_memory_dir
from langchain_core.messages.utils import count_tokens_approximately
from langchain_core.tools import tool

print("🤖 [Memory Agent 직접 조립 중...]")

# 1. LLM 초기화 (configs/model.config 파일 로드)
with open(os.path.join(project_root, "configs/model.config"), "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
llm = init_chat_model(
    model=model_cfg.get("model_name", "gemini-3.8-flash"),
    temperature=model_cfg.get("temperature", 0.0)
)

# 2. 스토어 및 미들웨어 초기화
#    - Semantic: 기본 스토어를 통합 워크스페이스(artifacts/users/default/memory)로 지정하면,
#      MemoryMiddleware가 user_id마다 artifacts/users/{user_id}/memory/ 스토어를 따로 엽니다.
#    - Episodic: 실습 샌드박스 DB 하나를 공유하되 user_id 컬럼으로 사용자별 격리됩니다.
NOTEBOOK_USER = "notebook_temp_user"
shutil.rmtree(USERS_ROOT_DIR / NOTEBOOK_USER, ignore_errors=True)   # 이전 실행 흔적 제거
user_mem_dir = resolve_user_memory_dir(NOTEBOOK_USER, ensure_exists=True)
for fname in ("USER.md", "MEMORY.md"):   # Part 2에서 다듬은 실습 프로필을 이 유저의 장기 기억으로 복사
    shutil.copy(os.path.join(demo_mem_dir, fname), user_mem_dir / fname)
print(f"👤 실습 유저 장기 기억: {user_mem_dir.relative_to(USERS_ROOT_DIR.parent.parent)}")

sem_store = SemanticMemoryStore(memory_dir=str(resolve_user_memory_dir("default")))
sem_store.load_from_disk()

epi_store = EpisodicStore(db_path=demo_epi_path)
await epi_store.setup()

memory_mw = MemoryMiddleware(semantic_store=sem_store, episodic_store=epi_store, review_llm=llm)

# 실습용 대형 도구: 2,000행짜리 매출 표 (약 7만 자)를 반환
@tool
def fetch_sales_report(quarter: str) -> str:
    """Fetch the full sales report table for a quarter (e.g. 'Q3'). Returns a very large text table."""
    regions = ['Seoul', 'Busan', 'Incheon', 'Daegu']
    rows = [f"row {i:05d} | region={regions[i % 4]} | revenue={i * 7} | quarter={quarter}" for i in range(2000)]
    return "\n".join(rows)

all_tools = list(tools_chatbot) + memory_mw.get_tools() + [fetch_sales_report]

# 3. 5-Layer Prompt Middleware 구성
skill_builder = SkillPromptBuilder(
    skills_dirs=[os.path.join(project_root, "skills"), os.path.join(project_root, ".agents/skills")],
    guidelines_path=os.path.join(project_root, "app/prompts/SKILL.md") if os.path.exists(os.path.join(project_root, "app/prompts/SKILL.md")) else None,
)
assembler = PromptAssembler(
    system_rules="You are an advanced AI Software Engineer equipped with long-term memory and precise reasoning capabilities.",
    tool_schemas=all_tools,
    skill_catalog=skill_builder.assemble,
    agent_rules_path=os.path.join(project_root, "app/prompts/SKILL.md") if os.path.exists(os.path.join(project_root, "app/prompts/SKILL.md")) else None,
)
prompt_mw = create_prompt_assembler_middleware(assembler, merge_system=True)

# 4. Compaction 미들웨어 구성
#    trigger는 '대화 추정 토큰' 또는 '직전 모델 호출의 실사용 토큰(시스템 프롬프트·도구 스키마 포함)'과 비교됩니다.
#    이 에이전트의 5-Layer 시스템 프롬프트가 약 6,500 토큰이므로, 대화가 약 2,500 토큰 쌓이면 요약되도록 9,000으로 둡니다.
#    (실습용으로 작게 잡은 값. 실서비스는 모델 최대 입력의 70~80% 수준)
compaction_mws = create_compaction_middleware(
    llm,
    trigger_tokens=9000,      # 실사용 9,000 토큰을 넘으면 요약
    keep_tokens=800,          # 최근 800 토큰 분량의 대화는 원문 유지
    # swap_dir 미지정 → artifacts/users/{user_id}/sessions/{thread_id}/tool-results/ 에 저장
)
print(f"🗜️ Compaction 미들웨어: {[type(m).__name__ for m in compaction_mws]}")

# 5. Checkpointer 연결 (샌드박스 DB)
chk_conn = await aiosqlite.connect(demo_chk_path, check_same_thread=False)
checkpointer = AsyncSqliteSaver(chk_conn)
await checkpointer.setup()

# 6. create_agent로 에이전트 빌드!
agent = create_agent(
    model=llm,
    tools=all_tools,
    middleware=[memory_mw, *compaction_mws, prompt_mw],
    checkpointer=checkpointer,
    context_schema=AgentContext,
)

# 참조 저장 (실습 시나리오 및 리소스 정리용)
agent.episodic_store = epi_store
agent.semantic_store = sem_store
agent.checkpointer_conn = chk_conn

ctx = AgentContext(
    user_id=NOTEBOOK_USER,   # 👤 기억·워크스페이스·Swap 파일이 모두 이 유저 기준으로 분리됨
    semantic_memory_enabled=True,
    semantic_learning_enabled=True,
    episodic_memory_enabled=True,
    episodic_learning_enabled=True,
)

async def show_context(thread_id: str):
    """해당 스레드의 대화(state) 크기, 직전 모델 호출의 실사용 토큰, 요약 여부를 출력합니다."""
    msgs = (await agent.aget_state({"configurable": {"thread_id": thread_id}})).values["messages"]
    has_summary = any(is_compaction_summary(m) for m in msgs)
    last_ai = next((m for m in reversed(msgs) if isinstance(m, AIMessage) and m.usage_metadata), None)
    used = last_ai.usage_metadata["total_tokens"] if last_ai else 0
    print(f"📏 [{thread_id}] 대화 {len(msgs)}개 메시지(약 {count_tokens_approximately(msgs):,} 토큰) | "
          f"직전 호출 실사용 {used:,} 토큰 (시스템 프롬프트 포함, trigger 9,000) | "
          f"요약 {'있음 📝' if has_summary else '없음'}")

print("✅ create_agent를 통한 에이전트 조립 및 준비 완료!")

### 6-2. [시나리오 1] Semantic Memory 기반 개인화 프로필 QA (`USER.md` 반영)

In [ ]:
query_1 = "내 이름과 전문 분야, 그리고 내가 선호하는 설명 스타일을 알려줘."
print(f"👤 질문: {query_1}")

res_1 = await agent.ainvoke(
    {"messages": [HumanMessage(content=query_1)]},
    config={"configurable": {"thread_id": "scenario_01"}},
    context=ctx
)
ans_1 = normalize_content([m for m in res_1["messages"] if isinstance(m, AIMessage)][-1].content)
print(f"\n🤖 답변 (전문):\n{ans_1}")

### 6-3. [시나리오 2] Episodic 8개 대화 중 2-Stage JIT 회상 (`session_recall` Window 슬라이싱)

In [ ]:
# 1. 과거 세션 회의록 데이터 8개 메시지 저장 (설계 -> 정책 -> 저장소 논의)
past_sid = "session_jwt_auth_policy_2026"
session_jwt_msgs = [
    {"role": "human", "content": "오늘 마이크로서비스 인증 아키텍처 설계 회의를 시작합시다."},
    {"role": "ai", "content": "네, 세션 기반 인증과 JWT 토큰 인증 중 어떤 방식을 검토할까요?"},
    {"role": "human", "content": "확장성을 위해 Stateless JWT 인증 방식을 채택합시다."},
    {"role": "ai", "content": "좋습니다. Stateless JWT 아키텍처로 진행하겠습니다."},
    {"role": "human", "content": "JWT 토큰의 Access Token과 Refresh Token 유효기간 정책은 어떻게 되나요?"},
    {"role": "ai", "content": "Access Token은 15분, Refresh Token은 7일로 확정했습니다."},
    {"role": "human", "content": "Refresh Token은 보안상 어디에 저장할 계획인가요?"},
    {"role": "ai", "content": "XSS 공격 방지를 위해 클라이언트 저장소가 아닌 HTTP-Only Secure Cookie에 보관합니다."}
]
# user_id를 지정해 이 유저의 과거 세션으로 저장 (session_recall은 현재 유저의 세션만 열람 가능)
await agent.episodic_store.finalize_session(past_sid, session_jwt_msgs, llm=None, user_id=NOTEBOOK_USER)
print(f"💾 [과거 세션({past_sid}) 회의록 8개 메시지 저장 및 FTS5 인덱싱 완료]")

# 2. 유저 질문: 에이전트는 1단계 선제 요약 힌트를 확인한 뒤, 정밀 인출을 위해 session_recall 도구를 호출합니다.
query_2 = "이전에 우리가 정한 JWT 토큰 만료 정책(Access/Refresh 유효기간)이 뭐였지? session_recall 도구로 확인해줘."
print(f"\n👤 질문: {query_2}")

res_2 = await agent.ainvoke(
    {"messages": [HumanMessage(content=query_2)]},
    config={"configurable": {"thread_id": "scenario_02"}},
    context=ctx
)

# 3. 도구 호출 궤적 분석 (전체 8개 중 관련 Window 및 Bookend 인출 확인)
for m in res_2["messages"]:
    if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
        print(f"\n🎯 [에이전트 도구 호출]: {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})")
    elif isinstance(m, ToolMessage):
        print(f"\n🔧 [도구 반환 데이터 (전체 8개 중 Anchor Window 기반 발췌 전문)]:\n{m.content}")

ans_2 = normalize_content([m for m in res_2["messages"] if isinstance(m, AIMessage)][-1].content)
print(f"\n🤖 답변 (전문):\n{ans_2}")

### 6-4. [시나리오 3] memory 도구를 활용한 장기 기억 자율 쓰기

In [ ]:
query_3 = "다음을 기억해줘: 나의 최애 과일은 애플망고야. memory 도구의 add 액션으로 user 타겟에 저장해."
print(f"👤 지시: {query_3}")

res_3 = await agent.ainvoke(
    {"messages": [HumanMessage(content=query_3)]},
    config={"configurable": {"thread_id": "scenario_03"}},
    context=ctx
)

for m in res_3["messages"]:
    if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
        print(f"  🎯 [에이전트 도구 호출]: {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})")
    elif isinstance(m, ToolMessage):
        print(f"  🔧 [도구 반환 데이터 전문]: {m.content}")

ans_3 = normalize_content([m for m in res_3["messages"] if isinstance(m, AIMessage)][-1].content)
print(f"\n🤖 답변 (전문):\n{ans_3}")

### 6-5. [시나리오 4] 대형 도구 결과 Swap → 에이전트가 필요한 행만 찾아 읽기

`fetch_sales_report`는 약 7만 자의 표를 돌려줍니다. `ToolResultSwapMiddleware`가 이를 디스크에 저장하고 미리보기만 대화에 넣으면, 에이전트는 미리보기에 없는 행을 **`grep_search`/`file_read`로 Swap 파일에서 직접 찾아** 답합니다.

이 시나리오와 다음 시나리오는 같은 스레드(`scenario_04`)에서 이어집니다.

In [ ]:
query_4 = ("이번 분석의 프로젝트 코드명은 BLUE-HERON이야. "
           "Q3 매출 보고서를 가져와서 row 01234의 region과 revenue를 알려줘.")
print(f"👤 질문: {query_4}")

res_4 = await agent.ainvoke(
    {"messages": [HumanMessage(content=query_4)]},
    config={"configurable": {"thread_id": "scenario_04"}},
    context=ctx
)

for m in res_4["messages"]:
    if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
        for tc in m.tool_calls:
            print(f"\n🎯 [에이전트 도구 호출]: {tc['name']}({tc['args']})")
    elif isinstance(m, ToolMessage):
        body = str(m.content)
        print(f"🔧 [도구 반환 ({len(body):,}자)]: {body[:300]}{' ...' if len(body) > 300 else ''}")

ans_4 = normalize_content([m for m in res_4["messages"] if isinstance(m, AIMessage)][-1].content)
print(f"\n🤖 답변 (전문):\n{ans_4}")
print(f"\n✅ 정답: row 01234 → region=Incheon, revenue=8638")
await show_context("scenario_04")

### 6-6. [시나리오 5] 대화 누적 → 자동 요약 → 요약 이후에도 이전 사실 기억

같은 스레드에서 긴 답변이 필요한 질문을 몇 번 더 던져 대화를 키웁니다. 직전 모델 호출의 실사용 토큰(시스템 프롬프트 약 6,500 + 대화)이 `trigger_tokens`(9,000)를 넘으면, 다음 모델 호출 직전에 `SummarizationMiddleware`가 앞부분을 요약합니다. 매 턴 뒤 `show_context()`로 대화 크기, 실사용 토큰, 요약 여부를 확인하세요.

> 💡 `trigger`는 **대화 메시지 추정치**와 **직전 호출이 보고한 실사용량** 중 하나라도 넘으면 발동합니다. 실사용량에는 시스템 프롬프트와 도구 스키마가 포함되므로, trigger를 시스템 프롬프트보다 작게 잡으면 **매 호출마다 요약**이 일어납니다.

마지막 질문에서는 **도구 없이 대화 내용만으로** 첫 턴의 코드명과 매출 값을 답하게 해, 요약이 핵심 사실을 보존했는지 검증합니다.

In [ ]:
follow_ups = [
    "분기 매출 보고서를 읽을 때 꼭 봐야 할 지표 5가지를 각각 3문장씩 설명해줘.",
    "지역별 매출 편차가 클 때 원인을 분석하는 절차를 단계별로 자세히 설명해줘.",
    "다음 분기 매출 예측 모델을 만들 때 고려할 변수와 검증 방법을 자세히 설명해줘.",
]
for q in follow_ups:
    print(f"👤 질문: {q}")
    await agent.ainvoke(
        {"messages": [HumanMessage(content=q)]},
        config={"configurable": {"thread_id": "scenario_04"}},
        context=ctx
    )
    await show_context("scenario_04")
    print()

In [ ]:
# 1. 현재 state에 들어 있는 요약 메시지 확인
state_msgs = (await agent.aget_state({"configurable": {"thread_id": "scenario_04"}})).values["messages"]
summary_in_state = next((m for m in state_msgs if is_compaction_summary(m)), None)
if summary_in_state:
    print("📝 [state에 저장된 요약 메시지 (앞부분)]")
    print("─" * 80)
    print(str(summary_in_state.content)[:1500])
    print("─" * 80)
else:
    print("ℹ️ 아직 요약이 일어나지 않았습니다. 위 셀을 한 번 더 실행해 대화를 키워 보세요.")

# 2. 도구 없이 대화 내용만으로 첫 턴의 사실을 답하게 하기
query_5 = "도구를 쓰지 말고 지금까지의 대화 내용만으로 답해줘. 이번 분석의 프로젝트 코드명과 row 01234의 revenue는?"
print(f"\n👤 질문: {query_5}")
res_5 = await agent.ainvoke(
    {"messages": [HumanMessage(content=query_5)]},
    config={"configurable": {"thread_id": "scenario_04"}},
    context=ctx
)
ans_5 = normalize_content([m for m in res_5["messages"] if isinstance(m, AIMessage)][-1].content)
print(f"\n🤖 답변 (전문):\n{ans_5}")
print(f"\n✅ 기대값: BLUE-HERON / 8638")

### 6-7. 유저별 컨텍스트 격리 확인 (`artifacts/users/`)

Part 6의 시나리오는 모두 `user_id="notebook_temp_user"`로 실행했습니다. 실제 서버와 같은 규칙으로 기록이 나뉘어 쌓입니다.

```text
artifacts/users/notebook_temp_user/
├── memory/                         ← Semantic Memory (USER.md, MEMORY.md) — 6-4에서 memory 도구가 기록
└── sessions/
    └── scenario_04/
        └── tool-results/           ← 6-5에서 Swap된 대형 도구 결과
```
Episodic Memory(과거 세션 요약)는 DB 하나를 공유하되 `sessions.user_id` 컬럼으로 나뉩니다.

아래 셀에서 ① 실제 디렉토리와 DB를 확인하고, ② **같은 유저의 새 세션**에서는 6-4에서 저장한 기억을 꺼내 쓰며, ③ **다른 유저**는 같은 질문에 답하지 못하는 것을 확인합니다.

In [ ]:
def show_user_tree(user_id: str):
    """artifacts/users/{user_id} 아래 디렉토리와 파일을 트리로 출력합니다."""
    root = USERS_ROOT_DIR / user_id
    print(f"📂 {root.relative_to(USERS_ROOT_DIR.parent.parent)}/")
    for path in sorted(root.rglob("*")):
        depth = len(path.relative_to(root).parts)
        label = f"📄 {path.name}  ({path.stat().st_size:,} bytes)" if path.is_file() else f"📁 {path.name}/"
        print("   " + "    " * (depth - 1) + label)

# 1. 실습 유저의 디렉토리 트리
show_user_tree(NOTEBOOK_USER)

# 2. 6-4에서 memory 도구가 기록한 USER.md
print("\n📄 [USER.md 마지막 엔트리]")
print((user_mem_dir / "USER.md").read_text(encoding="utf-8").split("§")[-1].strip())

# 3. Episodic DB에서 유저별 세션 수
rows = await epi_store._conn.execute_fetchall("SELECT user_id, COUNT(*) FROM sessions GROUP BY user_id")
print("\n🗄️ [Episodic DB 유저별 세션 수]")
for uid, cnt in rows:
    print(f"   - {uid}: {cnt}개")

In [ ]:
question = "내 이름과 내가 제일 좋아하는 과일이 뭐였지? 도구는 쓰지 말고 알고 있는 기억만으로 답해줘."

# ② 같은 유저, 새 세션: 장기 기억(USER.md)은 세션을 넘어 유지됩니다.
res_same = await agent.ainvoke(
    {"messages": [HumanMessage(content=question)]},
    config={"configurable": {"thread_id": "scenario_07_same_user"}},
    context=ctx
)
print(f"👤 [{NOTEBOOK_USER}] 새 세션 답변:\n{normalize_content(res_same['messages'][-1].content)}\n")

# ③ 다른 유저: 같은 에이전트·같은 질문이어도 notebook_temp_user의 기억은 보이지 않습니다.
OTHER_USER = "notebook_other_user"
shutil.rmtree(USERS_ROOT_DIR / OTHER_USER, ignore_errors=True)
ctx_other = AgentContext(
    user_id=OTHER_USER,
    semantic_memory_enabled=True,
    semantic_learning_enabled=True,
    episodic_memory_enabled=True,
    episodic_learning_enabled=True,
)
res_other = await agent.ainvoke(
    {"messages": [HumanMessage(content=question)]},
    config={"configurable": {"thread_id": "scenario_07_other_user"}},
    context=ctx_other
)
print(f"👥 [{OTHER_USER}] 답변:\n{normalize_content(res_other['messages'][-1].content)}\n")

# 다른 유저는 빈 장기 기억 디렉토리를 새로 받습니다.
show_user_tree(OTHER_USER)

## 🎓 Part 7. [학습 정리] 프로덕션 운영 핵심 FAQ

### 📚 Memory

### Q1. Episodic Memory는 언제 업데이트되나요?
- **답변**: 매 턴 대화·도구 원문은 `transcript.db`에 즉시 기록되고, `after_agent` 훅에서 **`IdleReviewQueue`(15초 유휴 / 최대 30분)**에 예약되어 대화가 멈춘 뒤 백그라운드 스레드에서 `finalize_session()`(다중 주제 요약 및 FTS5 색인)을 실행합니다.
- 사용자에게 나가는 최종 응답에는 **0ms의 지연**을 주며, 원문은 훅이 이미 `transcript.db`에 기록해 두었으므로, 백그라운드 리뷰는 그 원문을 읽어 **주제별 요약(사용자 언어, 한/영 이중 키워드)**과 `episodic.db` FTS5 색인만 수행합니다.

### Q2. 왜 `session_search`를 도구로 주지 않고 `session_recall`만 주었나요?
- **답변**: **도구 피로(Tool Fatigue) 방지 및 프롬프트 캐싱 최적화** 때문입니다.
- 검색(Search)은 `MemoryMiddleware.before_agent()`가 유저 질문을 바탕으로 자동으로 수행하여 L4에 100자 힌트만 넣어줍니다.
- 에이전트는 이미 요약된 힌트를 보고, 정말 필요한 경우에만 `session_recall` 도구로 원본 메시지를 열람합니다. (2-Stage JIT 패턴)

### Q3. Semantic Memory의 'Frozen Snapshot'이 왜 중요한가요?
- **답변**: 세션 도중에 에이전트가 `memory(add)`로 새 기억을 추가했다고 해서 **현재 세션의 시스템 프롬프트가 매 턴마다 바뀌면 GPU KV-Cache가 모두 깨져버립니다.**
- 따라서 현재 세션에서는 세션 시작 시점의 스냅샷을 고정 사용하고, 디스크에만 즉시 저장해 둔 뒤 **다음 세션부터 최신 마크다운을 로드**합니다.

### 🗜️ Compaction

### Q4. `trigger`와 `keep`은 실서비스에서 어떻게 정하나요?
- **답변**: 실습에서는 요약을 눈으로 보려고 9,000 / 800 토큰처럼 작게 잡았습니다. 실서비스에서는 **모델 최대 입력의 70~80%**를 `trigger`로, 그 **10~30%**를 `keep`으로 둡니다.
- 모델 프로필(`max_input_tokens`)을 아는 모델이라면 `trigger=("fraction", 0.8)`, `keep=("fraction", 0.2)`처럼 비율로도 지정할 수 있습니다.
- `trigger`는 **대화 메시지 추정치**와 **직전 모델 호출의 실사용량**(`usage_metadata`, 시스템 프롬프트·도구 스키마 포함) 중 하나라도 넘으면 발동합니다. 시스템 프롬프트보다 작은 값을 주면 매 호출마다 요약이 일어납니다. 반면 `keep`의 cutoff는 대화 메시지 추정치로만 계산합니다.

### Q5. 요약 LLM 호출이 실패하면 어떻게 되나요?
- **답변**: 일시적 오류(rate limit, timeout)는 최대 3회까지 재시도합니다. 그래도 실패하면 **에러를 그대로 올리고 대화는 바꾸지 않습니다.** 에러 문구를 요약인 척 넣어 이전 대화를 잃어버리는 일이 없습니다.

### Q6. 오래된 도구 결과만 지우는 더 가벼운 방법은 없나요?
- **답변**: LangChain prebuilt **`ContextEditingMiddleware(edits=[ClearToolUsesEdit(...)])`**가 있습니다. 토큰이 기준을 넘으면 최근 N개를 제외한 오래된 도구 결과를 `[cleared]`로 비웁니다(LLM 호출 없음). Anthropic API의 `clear_tool_uses` 기능을 모델 무관하게 옮긴 것입니다. 요약보다 저렴하지만, 비운 결과는 다시 볼 수 없다는 점을 고려해 선택하세요.

## 🧹 Part 8. Clean-up & Reset (실습 리소스 초기화)

실습이 모두 완료된 후 생성되었던 **임시 샌드박스 디렉토리(`demo_dir`)와 열려있는 DB 커넥션, 실습 유저 디렉토리(`artifacts/users/notebook_*`)를 깔끔하게 종료 및 삭제**하여 다음 실습 때도 항상 깨끗한 상태에서 시작할 수 있도록 초기화합니다.

In [ ]:
# 0. 백그라운드 메모리 리뷰 중지 (15초 유휴 후 실행되는 리뷰가 삭제한 유저 디렉토리를 다시 만들지 않도록)
if 'memory_mw' in globals():
    memory_mw._review_queue.shutdown()
    for ev in list(memory_mw._review_cancel_events.values()):
        ev.set()
    memory_mw._review_executor.shutdown(wait=True, cancel_futures=True)

# 1. 열려있는 DB 커넥션 종료
if 'agent' in locals():
    if hasattr(agent, "checkpointer_conn"):
        await agent.checkpointer_conn.close()
    if hasattr(agent, "episodic_store"):
        await agent.episodic_store.close()

# 2. 실습 임시 샌드박스 디렉토리 완전 삭제
if 'demo_dir' in locals() and os.path.exists(demo_dir):
    shutil.rmtree(demo_dir, ignore_errors=True)
    print(f"🧹 실습 임시 샌드박스 삭제 완료: {demo_dir}")

# 3. Part 6 실습 유저 디렉토리 삭제 (artifacts/users/notebook_*)
for uid in [globals().get("NOTEBOOK_USER"), globals().get("OTHER_USER")]:
    if uid and (USERS_ROOT_DIR / uid).exists():
        shutil.rmtree(USERS_ROOT_DIR / uid, ignore_errors=True)
        print(f"🧹 실습 유저 디렉토리 삭제 완료: artifacts/users/{uid}")

# 4. 혹시 남아있을 수 있는 노트북 하위 잔여 폴더 정리
for stale in ["app", "artifacts"]:
    p = os.path.join(current_dir, stale)
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)

print("✨ 모든 실습 리소스가 성공적으로 초기화되었습니다! (Ready for next run)")